# Transcribe Dübendorf Gemeinderat meetings on a free Colab GPU

The code comes from GitHub. Only the **data** lives on Google Drive (`MyDrive/DemocrAI-data`), so transcripts survive a disconnect.

**How to run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*, and allow Drive access.

The notebook first tries to download the meeting audio from YouTube itself, straight into Drive.
**Only if YouTube blocks Colab** ("Sign in to confirm you're not a bot"): upload the `.m4a` files from
`duebendorf-gr-rag/data/audio/` on your computer into `MyDrive/DemocrAI-data/audio/` and *Run all* again.

A T4 runs this model at roughly 20–40× realtime, so the ~69 h of audio takes about 2–4 h.
Keep the tab open. If Colab disconnects, just *Run all* again: finished meetings are skipped and an
unfinished one resumes from its last checkpoint (saved every 2 minutes).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU. Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", gpu.stdout.strip())

### Get the code and install

In [ ]:
REPO = "https://github.com/solomonhailedereje/DemocrAI.git"
CODE = "/content/DemocrAI"
import os
if os.path.isdir(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO} {CODE}
%cd {CODE}
!pip -q install -e ".[transcribe,convert]"

### Connect the Drive data folder
`data/` and `models/` in the repo become links to Drive, so everything written lands there.

In [ ]:
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive")
DATA = DRIVE / "DemocrAI-data"
MODELS = DRIVE / "DemocrAI-models"      # converted model is cached here, so it's built only once

assert (DATA / "meetings.json").exists(), f"Upload meetings.json to {DATA} first"
(DATA / "audio").mkdir(exist_ok=True)
(DATA / "transcripts").mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

for name, target in (("data", DATA), ("models", MODELS)):
    link = Path(CODE) / name
    if link.is_symlink() or link.exists():
        if not link.is_symlink():
            raise SystemExit(f"{link} exists and is not a link - remove it and re-run")
        link.unlink()
    link.symlink_to(target)

print("Drive data folder connected:", DATA)

### Audio
Downloads only what is missing (~30 MB per meeting-hour), straight into Drive. One meeting (2023-04-03) is a private video and is skipped.

In [ ]:
!pip -q install -U yt-dlp
!gr-rag download-audio
from gr_rag.download import AUDIO_EXTS
audio = sorted(f for f in (DATA / "audio").iterdir() if f.suffix in AUDIO_EXTS)
print(f"{len(audio)} audio files on Drive, {sum(f.stat().st_size for f in audio)/1e9:.1f} GB")
assert audio, ("No audio: YouTube blocked Colab. Upload the .m4a files from data/audio/ on your "
               f"computer into {DATA/'audio'} and Run all again.")

### Convert the model to CTranslate2 (one-off, ~5 min)
Skipped automatically if `MyDrive/DemocrAI-models` already holds it.

In [ ]:
!gr-rag convert-model

### Transcribe
float16 on the GPU is picked automatically. Progress and ETA are printed every 2 minutes.

In [ ]:
!gr-rag transcribe

### Check the result

In [ ]:
import json
done, partial = [], []
for f in sorted((DATA / "transcripts").glob("*.json")):
    t = json.loads(f.read_text(encoding="utf-8"))
    (done if t.get("complete") else partial).append((f.stem, len(t["segments"])))
missing = [a.stem for a in audio if a.stem not in {m for m, _ in done + partial}]
print(f"complete: {len(done)}/{len(audio)}   partial: {len(partial)}   not started: {len(missing)}")
for mid, n in partial: print("  partial", mid, n, "segments")
for mid in missing: print("  missing", mid)
empty = [mid for mid, n in done if n == 0]
if empty: print("  WARNING complete but empty:", empty)
if len(done) == len(audio) and not empty:
    print("All meetings transcribed.")
else:
    print("Not finished yet - run the Transcribe cell again (it resumes).")

### Done: bring the transcripts home

Download `MyDrive/DemocrAI-data/transcripts/` into `duebendorf-gr-rag/data/transcripts/` on your computer
(the audio does not need to come back), then run there:

```bash
gr-rag status                              # every meeting with audio should show transcript=✓
gr-rag update --only index summaries       # build the search index + summaries (needs Ollama)
gr-rag serve
```